#### Data Ingestion 

#### Document

A Document is a text chunk with optional metadata. It has:
- **Content**: the text
- **Metadata**: optional key-value pairs (source, author, date, etc.)


In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from pathlib import Path

/Users/shreyanshsingh/Documents/PersonalProjects/Procore_P6_Automation/RAG KBs/rag_implementation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dir_pdf_loader = DirectoryLoader(
    "../raw_docs/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
documents=dir_pdf_loader.load()
print(documents)

[Document(metadata={'producer': 'Nuance PDF Create', 'creator': 'Microsoft Word - p6_eppm_data_dictionary.docx', 'creationdate': '2021-04-07T12:56:46-07:00', 'source': '../raw_docs/p6_eppm_data_dictionary.pdf', 'file_path': '../raw_docs/p6_eppm_data_dictionary.pdf', 'total_pages': 89, 'format': 'PDF 1.5', 'title': 'P6 Data Dictionary', 'author': 'Oracle Construction and Engineering', 'subject': '', 'keywords': 'Version 21', 'moddate': '2021-04-07T12:57:43-07:00', 'trapped': '', 'modDate': "D:20210407125743-07'00'", 'creationDate': "D:20210407125646-07'00'", 'page': 0}, page_content='Oracle \nPrimavera \nP6 Data Dictionary \nVersion 21 \nApril 2021'), Document(metadata={'producer': 'Nuance PDF Create', 'creator': 'Microsoft Word - p6_eppm_data_dictionary.docx', 'creationdate': '2021-04-07T12:56:46-07:00', 'source': '../raw_docs/p6_eppm_data_dictionary.pdf', 'file_path': '../raw_docs/p6_eppm_data_dictionary.pdf', 'total_pages': 89, 'format': 'PDF 1.5', 'title': 'P6 Data Dictionary', 'aut

In [3]:
num_files = len(set(doc.metadata.get("source", "") for doc in documents))
print(f"Unique PDF files: {num_files}")

Unique PDF files: 2


In [4]:
from typing import Any


def process_all_pdfs(path_pdf_directory):
    """ Process all PDFs in the directory """
    all_documents=[]
    pdf_dir = Path(path_pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf")) #Read all PDFs in the path
    print(f"Found {len(pdf_files)} PDF files")

    for each_file in pdf_files:
        print(f"Processing {each_file}")

        try:
            loader = PyPDFLoader(str(each_file)) # Load each PDF doc
            documents = loader.load() # Assign it in a variable documents

            for doc in documents: #Adding additional metadata to the Document({metadata: .... , page_content:....})
                doc.metadata['source_file'] = each_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" ✔ Loaded {len(documents)} pages")

        except Exception as e:
            print(f" ⨯ Error : {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

    

In [5]:
all_documents = process_all_pdfs("../raw_docs/")

Found 2 PDF files
Processing ../raw_docs/p6_eppm_data_dictionary.pdf
 ✔ Loaded 89 pages
Processing ../raw_docs/Construction RAG Knowledge Base_.pdf
 ✔ Loaded 37 pages

 Total documents loaded: 126


### Creating chunks
We will take each document and recursively split them in chunks

In [6]:
### Split text into Chunks

def split_docs_in_chunks(documents , chunk_size=1000 , chunk_overlap = 200 ):
    """ Split documents in Chunks of data """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                    chunk_overlap = chunk_overlap,
                                                    length_function = len,
                                                    separators=["\n\n", "\n", ". ", " ", ""])
    split_chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_chunks)} chunks")

    if split_chunks:
        print(f"Sample chunk --> \n")
        print(f"{split_chunks[0].page_content[:150]}")
        print(f"{split_chunks[0].metadata}")

    return split_chunks

In [7]:
chunks = split_docs_in_chunks(all_documents,1000,200)

Split 126 documents into 453 chunks
Sample chunk --> 

Oracle 
Primavera 
P6 Data Dictionary 
Version 21 
April 2021
{'producer': 'Nuance PDF Create', 'creator': 'Microsoft Word - p6_eppm_data_dictionary.docx', 'creationdate': '2021-04-07T12:56:46-07:00', 'moddate': '2021-04-07T12:57:43-07:00', 'author': 'Oracle Construction and Engineering', 'title': 'P6 Data Dictionary', 'keywords': 'Version 21', 'source': '../raw_docs/p6_eppm_data_dictionary.pdf', 'total_pages': 89, 'page': 0, 'page_label': '1', 'source_file': 'p6_eppm_data_dictionary.pdf', 'file_type': 'pdf'}


# Embedding the Chunks - 1

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple , Dict , Any
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """ Generates Embedding from text using SentenceTransformer """

    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """Initialize Embedding Manager
            Args:
                model_name : HuggingFace model for sentence embeddings
         """

        self.model_name = model_name
        self.model = None
        self._load_model()
    

    def _load_model(self):
        """ Load SentenceTransformer Model """

        try:
            print(f"Loading model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} , loaded succesfully ✅")


        except Exception as e:
            print(f" Error in loading model {self.model_name}: {e}")
            raise    

    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        """ Generate embeddings for list of text
                Args:
                    texts : List of text strings to embed

                Returns:
                    numpy array of embeddings 
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embedddings ✅")
        return embeddings


embedding_mgr = EmbeddingManager()
embedding_mgr

Loading model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2880.72it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model all-MiniLM-L6-v2 , loaded succesfully ✅


### Vector Store

In [10]:
class VectorStore:
    """ Manages ChromaDB Vector Store"""
    def __init__(self,collection_name:str= "construction_pdf_docs", persist_directory:str="../data/vector_store"):
        """ Initialise Vector Store
            
            Args:
                collection_name : Name of the ChromaDB collection
                persist_directory: Local fiolder where ChromaDB will be stored
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialise_store()
        

    def _initialise_store(self):
        """Initialise ChromaDB client and connection """

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Doc embeddings for RAG"},
            )
            print(f"Initilized vector store. Collection Name  : {self.collection_name}")
            print(f" Collection Docs Count  : {self.collection.count()}")
        except Exception as e:
            print(f"Error in creating Vector Store : {e}")
            raise
    
    def add_documents(self ,documents:List[Any], embeddings: np.ndarray):
        """ Add documents and their embeddingsto the vector store
        
            Args:
                documents : List of LangChain documents to add
                embeddings : numpy array of embeddings

            Returns:
                List of document ids    
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same")
        
        print(f"Adding {len(documents)} documents to the vector store...")
        # Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc , embedding ) in enumerate(zip(documents,embeddings)):
            # Generate unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

             # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try :
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
                )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore = VectorStore()
vectorstore

Initilized vector store. Collection Name  : construction_pdf_docs
 Collection Docs Count  : 2265


In [ ]:
# Convert text to embeddings
# Step - 1 : Get actual content from Documents --> documents is a list of LangChain documents with metadata and page_content
texts= [ doc.page_content for doc in chunks]
texts

# Step: - 2 : Generate Embeddings
embeddings = embedding_mgr.generate_embeddings(texts)

# Step - 3 : Add documents to Vector Store 
#            Uncomment the line below to add documents to vector store,
#             otherwise it will duplicate the documents in vector store every time you run the cell
# vectorstore.add_documents(chunks,embeddings)

Generating embeddings...


Batches: 100%|██████████| 15/15 [00:02<00:00,  6.17it/s]


Generated embedddings ✅
Adding 453 documents to the vector store...
Successfully added 453 documents to vector store
Total documents in collection: 2718


#### In case we may have duplicated the documents, use below code to check for duplicates

In [13]:
# Check for duplicate documents by content hash
def check_duplicate_documents(vectorstore: VectorStore):
    """Check for duplicate documents in the vector store"""
    try:
        # Get all documents from collection
        all_docs = vectorstore.collection.get(include=["documents", "metadatas"])
        
        documents = all_docs['documents']
        metadatas = all_docs['metadatas']
        ids = all_docs['ids']
        
        # Create a dictionary to track content hashes
        content_hashes = {}
        duplicates = []
        
        for doc_id, content, metadata in zip(ids, documents, metadatas):
            # Create a hash of content + key metadata
            content_hash = hash(content)
            
            if content_hash in content_hashes:
                duplicates.append({
                    "duplicate_id": doc_id,
                    "original_id": content_hashes[content_hash]["id"],
                    "source": metadata.get('source_file', 'unknown'),
                    "page": metadata.get('page', 'unknown')
                })
            else:
                content_hashes[content_hash] = {"id": doc_id, "content": content[:100]}
        
        print(f"Total documents: {len(documents)}")
        print(f"Unique documents: {len(content_hashes)}")
        print(f"Duplicate documents found: {len(duplicates)}")
        
        if duplicates:
            print("\nDuplicate details:")
            for dup in duplicates[:10]:  # Show first 10
                print(f"  Duplicate ID: {dup['duplicate_id']}, Original ID: {dup['original_id']}, Source: {dup['source']}")
        
        return duplicates
    except Exception as e:
        print(f"Error checking duplicates: {e}")
        raise

# Run the check
duplicates = check_duplicate_documents(vectorstore)

Total documents: 2718
Unique documents: 453
Duplicate documents found: 2265

Duplicate details:
  Duplicate ID: doc_d703a994_0, Original ID: doc_eb5b8285_0, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_6c6f5c95_1, Original ID: doc_1de0e6aa_1, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_2f4a5bac_2, Original ID: doc_5d699846_2, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_0ae7ff56_3, Original ID: doc_4bd51a96_3, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_61f1a71a_4, Original ID: doc_3f986371_4, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_fd545bf8_5, Original ID: doc_7507d74a_5, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_123d6832_6, Original ID: doc_d72a1a6c_6, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_f5b4246c_7, Original ID: doc_b793ef94_7, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_10125601_8, Original ID: doc_f0e0e836_8, Source: p6_eppm_data_dictionary.pdf
  Duplicate ID: doc_7d7db372_9,

In [ ]:
# Reset the collection
vectorstore.client.delete_collection(name=vectorstore.collection_name)
vectorstore.collection = vectorstore.client.get_or_create_collection(
    name=vectorstore.collection_name,
    metadata={"description": "PDF Doc embeddings for RAG"},
)
print(f"Collection reset. New count: {vectorstore.collection.count()}")

# Then reload documents fresh
embeddings = embedding_mgr.generate_embeddings([doc.page_content for doc in chunks])
# vectorstore.add_documents(chunks, embeddings)

Collection reset. New count: 0
Generating embeddings...


Batches: 100%|██████████| 15/15 [00:02<00:00,  6.57it/s]


Generated embedddings ✅
Adding 453 documents to the vector store...
Successfully added 453 documents to vector store
Total documents in collection: 453


### RAG Retriever Pipeline to get data from VectorStore as "CONTEXT"

In [19]:
# Inside retrieve_context()
class RAGRetriever:
    """Retrieves Context from Vector Store using RAG"""

    def __init__(self,vectorstore:VectorStore , embedding_mgr:EmbeddingManager):
        """Initialize RAGRetriever
            Args:
                vectorstore : VectorStore object
                embedding_mgr : EmbeddingManager object 
        """
        self.vectorstore = vectorstore
        self.embedding_mgr = embedding_mgr
    
    def retrieve_context(self,query:str,top_k:int=5,score_threshold : float = 0.5):
        """Retrieve Context from Vector Store
            Args:
                query : Query string
                top_k : Number of results to return
                score_threshold : Minimum similarity score for results

            Returns:
                List of documents and their similarity scores
        """
        # Generate Embeddings for query
        query_embedding = self.embedding_mgr.generate_embeddings([query])[0]

        try:
            # Query Vector Store
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances", "embeddings"]  # Make sure embeddings are included
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                doc_embeddings = results['embeddings'][0]  # Get document embeddings
                ids = results['ids'][0]

                # Compute cosine similarities
                similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

                for i, (doc, metadata, sim, id) in enumerate(zip(documents, metadatas, similarities, ids)):
                    

                    if sim > score_threshold:
                        retrieved_docs.append({
                            "content": doc,
                            "metadata": metadata,
                            "id": id,
                            "similarity_score": sim
                        })
                        # print(f"Retrieved Document {len(retrieved_docs)} with similarity score >= {sim}")

            else:
                print("No results found in Vector Store")

            return retrieved_docs
        except Exception as e:
            print(f"Error in retrieving context from Vector Store: {e}")
            raise
rag_retriever = RAGRetriever(vectorstore,embedding_mgr)
# rag_retriever.retrieve_context("RISK-FIN-001: Cost Overruns")

                        
        



### Provide Extracted context to LLM

In [20]:
def rag_simple(query , rag_retriever:RAGRetriever , top_k=5 ):
    """Simple RAG implementation to retrieve context for a query"""
    retrieved_docs = rag_retriever.retrieve_context(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in retrieved_docs]) if retrieved_docs else ""

    if not context:
        print("No relevant context found for the query.")
    return  context

In [31]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

#Initiate LLM :
llm_ollama = ChatOllama(model="gpt-oss:20b" , temperature=0.2)

In [ ]:

#Get the question from user to ask the LLM :
question = input("Enter your question: ")

#Initialise the role of the LLM in the prompt template
role = "helpful assistant who provides accurate and concise answers based on the provided context in Construction Project Management domain"
instructions = """The response should be formatted as a Markdown list with clear headings and concise sentences."""

#Get context from RAG Retriever
rag_context = rag_simple(question, rag_retriever, top_k=5)

#Generate prompt for LLM using PromptTemplate
prompt_template = PromptTemplate(
    input_variables=["role", "question", "rag_context", "instructions"],
    template="""
You are a {role}. Use the following context to answer the following question:

Context: 
{rag_context}

Question:
{question}

Instructions:
{instructions}

Answer:
"""
)

#Set the prompt with the question and retrieved context from RAG Retriever
formatted_prompt = prompt_template.format(role="helpful assistant", question=question, instructions=instructions, rag_context="")

#Use formatted prompt to get response from LLM
response = llm_ollama.invoke(formatted_prompt)

print(f"Question: {question}\n")
print(response)

Generating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.33it/s]

Generated embedddings ✅
No relevant context found for the query.


Question: What is EPPM dictionary

content="```markdown\n# EPPM Dictionary\n\n- **EPPM**: Stands for Enterprise Project Portfolio Management.\n- **Definition**: A comprehensive system or process used by organizations to manage their projects, programs, and portfolios in an integrated manner.\n- **Purpose**: To ensure that all project-related activities are aligned with the organization's strategic goals, resources are efficiently allocated, and risks are effectively managed.\n```" additional_kwargs={} response_metadata={'model': 'qwen2.5-coder:7b', 'created_at': '2026-03-04T22:43:39.919377Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4531535125, 'load_duration': 71667958, 'prompt_eval_count': 77, 'prompt_eval_duration': 441800500, 'eval_count': 82, 'eval_duration': 3988520330, 'logprobs': None, 'model_name': 'qwen2.5-coder:7b', 'model_provider': 'ollama'} id='lc_run--019cbb05-575a-7a73-af44-c560f268c20e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens':

In [24]:
def rag_advanced(query, rag_retriever: RAGRetriever, top_k=5):
    """Advanced RAG implementation that retrieves context and returns metadata about sources and confidence scores"""
    retrieved_docs = rag_retriever.retrieve_context(query, top_k=top_k)
    
    context = "\n\n".join([doc["content"] for doc in retrieved_docs]) if retrieved_docs else ""
    
    # Extract source and confidence information
    sources_and_scores = []
    if retrieved_docs:
        for doc in retrieved_docs:
            sources_and_scores.append({
                "source": doc["metadata"].get("source_file", "unknown"),
                "page": doc["metadata"].get("page", "unknown"),
                "confidence_score": round(doc["similarity_score"], 4)
            })
    
    if not context:
        print("No relevant context found for the query.")
    
    return {
        "context": context,
        "sources": sources_and_scores,
        "num_documents_retrieved": len(retrieved_docs)
    }

In [ ]:
#Get the question from user to ask the LLM :
question = input("Enter your question: ")

#Initialise the role of the LLM in the prompt template
role = "helpful assistant who provides accurate and concise answers based on the provided context in Construction Project Management domain"
instructions = """The response should be formatted as a Markdown list with clear headings and concise sentences."""

#Get context from RAG Retriever
rag_context = rag_advanced(question, rag_retriever, top_k=5)

#Generate prompt for LLM using PromptTemplate
prompt_template = PromptTemplate(
    input_variables=["role", "question", "rag_context", "instructions"],
    template="""
You are a {role}. Use the following context to answer the following question:

Context: 
{rag_context}

Question:
{question}

Instructions:
{instructions}

Answer:
"""
)

#Set the prompt with the question and retrieved context from RAG Retriever
formatted_prompt = prompt_template.format(role=role, question=question, instructions=instructions, rag_context="")

#Use formatted prompt to get response from LLM
response = llm_ollama.invoke(formatted_prompt)

print(f"Question: {question}\n")
print(response.content,truncate=False)

print("Sources and Confidence Scores for retrieved context:")
for source in rag_context["sources"]:
    print(f"Source: {source['source']} (Page: {source['page']}), Confidence Score: {source['confidence_score']}")   

Generating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.80it/s]

Generated embedddings ✅


Question: What are common risks in Datacenter construction

**Common Risks in Datacenter Construction**

### 1. **Delays in Permitting and Approvals**

*   Delays in obtaining necessary permits and approvals can impact project timelines.
*   Inadequate planning and communication with regulatory bodies can exacerbate these risks.

### 2. **Supply Chain Disruptions**

*   Shortages or delays in delivery of critical components, such as servers or cooling systems, can hinder progress.
*   Global supply chain disruptions can further complicate matters.

### 3. **Weather-Related Delays**

*   Inclement weather conditions, such as extreme temperatures or flooding, can impact construction schedules and costs.
*   Datacenter construction often requires specialized equipment and materials, making it vulnerable to weather-related delays.

### 4. **Cybersecurity Risks**

*   Inadequate security measures during construction can compromise the datacenter's overall security posture.
*   Unauthorized 